# 第一阶段 步骤02：创建变量的函数

> 来源：《深度学习入门2：自制框架》（斋藤康毅 著，郑明智 译，人民邮电出版社 2023）
> 目标：从零构建深度学习框架 **DeZero** 的第二步。

---

## 核心目标

在步骤01的 `Variable`（数据的"箱子"）基础上，引入 **`Function` 类**：把"箱子"升级成能进行计算的"魔法箱子"。

`Function` 是连接 `Variable` 的机制——输入 `Variable`，输出 `Variable`。

## 2.1 什么是函数

函数是**连接 `Variable` 实例（数据"箱子"）的机制**，把普通"箱子"变成能进行计算的"魔法箱子"。

| 要点 | 说明 |
| --- | --- |
| 输入是 Variable | 函数的入参是 `Variable` 实例 |
| 输出是 Variable | 函数的结果也包装成 `Variable` 返回 |
| 数据在 `data` 里 | 实际数据仍存在实例变量 `data` 中 |

## 2.2 Function 类的实现

**初版**：把平方计算直接写死，用 `__call__` 让实例可以像函数一样被调用（`f(...)`）。

但这样每种计算都要重写一个类，不便复用。于是**重新设计**，定下两点要求：

1. **`Function` 是基类** —— 实现所有函数通用的流程；
2. **具体函数在子类中实现** —— 把计算逻辑下沉到 `forward`。

In [ ]:
import numpy as np

# 承接步骤01的 Variable
class Variable:
    def __init__(self, data):
        self.data = data

# 初版：把"平方"计算写死
class Function:
    def __call__(self, input):
        x = input.data       # 1) 取出数据
        y = x ** 2           # 2) 实际计算（写死为平方）
        output = Variable(y)  # 3) 包装成 Variable 返回
        return output

In [ ]:
# 最终版：Function 作为通用基类
class Function:
    def __call__(self, input):
        x = input.data          # 从 Variable 中取出数据
        y = self.forward(x)     # 具体计算交给 forward
        output = Variable(y)    # 结果包装成新的 Variable
        return output

    def forward(self, x):
        raise NotImplementedError()  # 提示：应由子类实现

## 2.3 使用 Function 类

只需继承 `Function` 并在 `forward` 里写计算逻辑，就得到一个具体函数。

`__call__` 承担通用流程（取数据 → 调 `forward` → 包装返回），所以子类只需专注 `forward`。

In [ ]:
# 继承 Function，实现平方函数（依赖上面已定义的 Variable 和 Function）
class Square(Function):
    def forward(self, x):
        return x ** 2

x = Variable(np.array(10))
f = Square()
y = f(x)

print(type(y))    # <class '__main__.Variable'>
print(y.data)     # 100

## 其他要点

- 步骤02 **只处理「单一输入、单一输出」**的函数；多变量支持从步骤11才开始扩展。
- 这种"统一基类 + 子类实现"的设计与 **PyTorch 的 `Function` 类**思路一致，便于后续统一管理、扩展（如反向传播）。

## 这一步的"为什么"

把"通用流程"和"具体计算"分离，是后续自动微分的关键铺垫：

- `__call__` 是唯一入口，未来在这里统一插入**保存输入、调用反向传播**等逻辑；
- 新增函数（`Sin`、`Exp`…）只需继承并写 `forward`，框架本身不用改。

---

> 预告：步骤06 会给 `Function` 加上 `backward` 方法、并让 `__call__` 保存输入变量（`self.input`），为反向传播做准备。